# VLM Temporal Operation Intelligence — Kaggle Fine-Tuning Notebook

## Setup Instructions (Read Before Running)

1. **Enable GPU**: Settings > Accelerator > GPU T4 x2
2. **Add HuggingFace Token**: Settings > Secrets > Add `HF_TOKEN` with your HuggingFace token
3. **Upload Training Data**: Create a Kaggle Dataset with your `train.json` and `val.json`, then add it as input
4. **Internet Access**: Settings > Internet > Enable (needed to download the model)

**Model:** Qwen2.5-VL-2B-Instruct (downloaded from HuggingFace at runtime, ~4GB)
**Compute:** Kaggle 2x T4 (16GB VRAM each)
**Method:** QLoRA 4-bit fine-tuning with LoRA rank 64
**Task:** Operation classification + temporal grounding + next-operation anticipation

In [ ]:
# ==========================================
# VRAM MATH (Required Format)
# ==========================================

# VRAM Budget Calculation
model_base_4bit    = 2.0   # GB — Qwen2.5-VL-2B at 4-bit quantization
lora_adapters      = 0.3   # GB — LoRA rank 64, alpha 128
frames_per_clip    = 8     # Sampled frames per 5-sec clip
frame_tokens       = 256   # Visual tokens per frame (after vision encoder)
batch_size         = 2     # Per-device batch size
token_hidden_dim   = 1536  # Qwen2-VL-2B hidden size

activation_gb      = (frames_per_clip * frame_tokens * batch_size * token_hidden_dim * 2) / 1e9
print(f"Raw activation memory: {activation_gb:.2f} GB")

# With gradient checkpointing: activation_gb * 0.4 (recomputed, not stored)
checkpointed_activation_gb = activation_gb * 0.4
print(f"Checkpointed activation memory: {checkpointed_activation_gb:.2f} GB")

total_vram_gb      = model_base_4bit + lora_adapters + checkpointed_activation_gb
print(f"Estimated VRAM: {total_vram_gb:.2f} GB")
print(f"T4 headroom: {16 - total_vram_gb:.2f} GB")

assert total_vram_gb < 16, f"VRAM budget {total_vram_gb:.2f} GB exceeds T4 16GB!"
print("✓ VRAM budget fits within T4 16GB")

In [ ]:
# ==========================================
# 1. Environment Setup (Kaggle)
# ==========================================

import os

# --- Kaggle-specific: Authenticate with HuggingFace ---
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("HuggingFace token loaded from Kaggle Secrets")
except Exception:
    print("WARNING: No HF_TOKEN found in Kaggle Secrets.")
    print("Add it via: Settings > Secrets > Add 'HF_TOKEN'")
    print("Get your token at: https://huggingface.co/settings/tokens")

# --- Install dependencies ---
!pip install -q --upgrade pip
!pip install -q transformers>=4.40.0 accelerate>=0.28.0
!pip install -q bitsandbytes>=0.43.0 peft>=0.10.0 trl>=0.8.0
!pip install -q qwen-vl-utils decord opencv-python-headless
!pip install -q Pillow scipy scikit-learn tqdm

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Enable GPU: Settings > Accelerator > GPU T4 x2")

In [ ]:
# ==========================================
# 2. Configuration (Kaggle T4 Optimized)
# ==========================================

from dataclasses import dataclass
import os

# --- Auto-detect Kaggle input datasets ---
KAGGLE_INPUT = "/kaggle/input"
dataset_dir = None
if os.path.exists(KAGGLE_INPUT):
    datasets = os.listdir(KAGGLE_INPUT)
    print(f"Available Kaggle datasets: {datasets}")
    for ds in datasets:
        ds_path = os.path.join(KAGGLE_INPUT, ds)
        if os.path.isfile(os.path.join(ds_path, "train.json")):
            dataset_dir = ds_path
            break
    if dataset_dir:
        print(f"Found training data in: {dataset_dir}")
    else:
        print("WARNING: No dataset with train.json found.")
        print("Upload your training data as a Kaggle Dataset and add it as input.")

@dataclass
class Config:
    # Model (downloaded from HuggingFace at runtime, not stored in repo)
    model_id: str = "Qwen/Qwen2-VL-2B-Instruct"
    
    # LoRA
    lora_rank: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    
    # Training (tuned for T4 16GB)
    num_epochs: int = 3
    per_device_batch_size: int = 2
    gradient_accumulation_steps: int = 8  # Effective batch size = 16
    learning_rate: float = 2e-4
    vision_lr: float = 2e-5  # 10x smaller for vision encoder
    warmup_ratio: float = 0.1
    max_seq_length: int = 2048
    
    # Video
    frames_per_clip: int = 8
    frame_size: int = 336  # Qwen2.5-VL native resolution
    
    # Checkpointing
    save_steps: int = 50
    save_total_limit: int = 2  # Keep only 2 checkpoints to save disk
    
    # Kaggle Paths
    train_data: str = os.path.join(dataset_dir, "train.json") if dataset_dir else "/kaggle/input/YOUR-DATASET/train.json"
    val_data: str = os.path.join(dataset_dir, "val.json") if dataset_dir else "/kaggle/input/YOUR-DATASET/val.json"
    output_dir: str = "/kaggle/working/checkpoints"
    
config = Config()
print(f"\nConfig Summary:")
print(f"  Model: {config.model_id}")
print(f"  Train data: {config.train_data}")
print(f"  Val data: {config.val_data}")
print(f"  Output: {config.output_dir}")
print(f"  Effective batch size: {config.per_device_batch_size * config.gradient_accumulation_steps}")
print(f"  LoRA: rank={config.lora_rank}, alpha={config.lora_alpha}")

In [ ]:
# ==========================================
# 3. Load Model with 4-bit Quantization
# ==========================================

from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Downloading model: {config.model_id}")
print("This may take 3-5 minutes on first run...")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    config.model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    token=os.environ.get("HF_TOKEN"),
)

processor = AutoProcessor.from_pretrained(
    config.model_id,
    token=os.environ.get("HF_TOKEN"),
)

# Enable gradient checkpointing (critical for fitting video training in T4 VRAM)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

print(f"\nModel loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory remaining: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

In [ ]:
# ==========================================
# 4. Configure LoRA Adapters
# ==========================================

from peft import LoraConfig, get_peft_model, TaskType

# Target modules for Qwen2-VL
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=config.lora_rank,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=target_modules,
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable_params / 1e6:.2f}M")
print(f"LoRA adapter size estimate: {trainable_params * 2 / 1e9:.3f} GB")

In [ ]:
# ==========================================
# 5. Dataset Preparation
# ==========================================

import json
from torch.utils.data import Dataset
from PIL import Image
import os


class OpenPackVLMDataset(Dataset):
    """Dataset for VLM fine-tuning on OpenPack temporal data."""
    
    def __init__(self, data_path, processor, max_frames=8):
        with open(data_path, "r") as f:
            self.data = json.load(f)
        self.processor = processor
        self.max_frames = max_frames
        self.data_dir = os.path.dirname(data_path)
        print(f"Loaded {len(self.data)} training examples")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        conversations = item["conversations"]
        
        # Load video frames
        video_path = os.path.join(self.data_dir, item.get("video", ""))
        frames = self._load_frames(video_path)
        
        # Build chat messages
        user_msg = conversations[0]["value"]
        assistant_msg = conversations[1]["value"]
        
        # Build content with images
        image_content = [{"type": "image", "image": frame} for frame in frames]
        
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a warehouse operations analyst. Analyze video frames "
                    "and respond with JSON containing dominant_operation, "
                    "temporal_segment, anticipated_next_operation, and confidence."
                ),
            },
            {
                "role": "user",
                "content": image_content + [{"type": "text", "text": user_msg.replace("<video>\n", "")}],
            },
            {
                "role": "assistant",
                "content": assistant_msg,
            },
        ]
        
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        
        inputs = self.processor(
            text=[text],
            images=frames,
            padding="max_length",
            max_length=config.max_seq_length,
            truncation=True,
            return_tensors="pt",
        )
        
        # Squeeze batch dimension
        return {k: v.squeeze(0) for k, v in inputs.items()}
    
    def _load_frames(self, video_path):
        """Load frames from video file or frame directory."""
        frames = []
        
        # Try loading from video
        if os.path.exists(video_path):
            try:
                from decord import VideoReader, cpu
                vr = VideoReader(video_path, ctx=cpu(0))
                indices = list(range(0, len(vr), max(1, len(vr) // self.max_frames)))
                indices = indices[:self.max_frames]
                frames = [Image.fromarray(vr[i].asnumpy()) for i in indices]
            except Exception:
                pass
        
        # Try loading from frame directory
        if not frames:
            frame_dir = video_path.replace(".mp4", "_frames")
            if os.path.isdir(frame_dir):
                frame_files = sorted(
                    [f for f in os.listdir(frame_dir) if f.endswith(".jpg")]
                )[:self.max_frames]
                frames = [
                    Image.open(os.path.join(frame_dir, f)).convert("RGB")
                    for f in frame_files
                ]
        
        # Fallback: generate placeholder frames
        if not frames:
            frames = [
                Image.new("RGB", (336, 336), color=(128, 128, 128))
                for _ in range(self.max_frames)
            ]
        
        # Resize to target
        frames = [
            f.resize((config.frame_size, config.frame_size)) for f in frames
        ]
        
        return frames


# Create datasets
train_dataset = OpenPackVLMDataset(config.train_data, processor, config.frames_per_clip)
val_dataset = OpenPackVLMDataset(config.val_data, processor, config.frames_per_clip)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# ==========================================
# 6. Training Configuration
# ==========================================

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.per_device_batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    gradient_checkpointing=True,
    
    # Learning rate
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    lr_scheduler_type="cosine",
    
    # Precision
    fp16=True,
    bf16=False,  # T4 doesn't support bf16
    
    # Checkpointing
    save_strategy="steps",
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    
    # Logging
    logging_steps=10,
    logging_dir=os.path.join(config.output_dir, "logs"),
    report_to="none",  # Use "wandb" if configured
    
    # Memory optimization
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    
    # Resume support
    resume_from_checkpoint=True,
    
    # Optimizer
    optim="adamw_bnb_8bit",  # 8-bit Adam for memory savings
)

print("Training arguments configured")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
# ==========================================
# 7. Custom Data Collator
# ==========================================

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class VLMDataCollator:
    """Collates VLM training samples with proper padding."""
    
    processor: object
    
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # Stack all tensors
        batch = {}
        for key in features[0].keys():
            tensors = [f[key] for f in features]
            if isinstance(tensors[0], torch.Tensor):
                # Pad to same length if needed
                max_len = max(t.shape[0] for t in tensors)
                padded = []
                for t in tensors:
                    if t.shape[0] < max_len:
                        pad_size = max_len - t.shape[0]
                        if t.dim() == 1:
                            t = torch.nn.functional.pad(t, (0, pad_size), value=0)
                        else:
                            t = torch.nn.functional.pad(t, (0, 0, 0, pad_size), value=0)
                    padded.append(t)
                batch[key] = torch.stack(padded)
            else:
                batch[key] = tensors
        
        # Create labels (same as input_ids, -100 for padding)
        if "input_ids" in batch:
            labels = batch["input_ids"].clone()
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            batch["labels"] = labels
        
        return batch

In [ ]:
# ==========================================
# 8. Train!
# ==========================================

import gc

# Clear any cached memory before training
torch.cuda.empty_cache()
gc.collect()

data_collator = VLMDataCollator(processor=processor)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# Log GPU memory before training
print(f"GPU memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# Start training
print("\n" + "=" * 50)
print("Starting fine-tuning...")
print(f"Kaggle GPU quota: ~30 hrs/week")
print(f"Estimated time: depends on dataset size")
print("=" * 50)

checkpoint_path = None
if os.path.exists(config.output_dir):
    existing = [d for d in os.listdir(config.output_dir) if d.startswith("checkpoint")]
    if existing:
        checkpoint_path = os.path.join(config.output_dir, sorted(existing)[-1])
        print(f"Resuming from checkpoint: {checkpoint_path}")

train_result = trainer.train(resume_from_checkpoint=checkpoint_path)

# Log training metrics
print(f"\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ==========================================
# 9. Save LoRA Adapter + Download from Kaggle
# ==========================================

import shutil

adapter_dir = os.path.join(config.output_dir, "final_adapter")
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)

print(f"LoRA adapter saved to: {adapter_dir}")
print("\nSaved files:")
total_size = 0
for f in sorted(os.listdir(adapter_dir)):
    size_mb = os.path.getsize(os.path.join(adapter_dir, f)) / 1e6
    total_size += size_mb
    print(f"  {f}: {size_mb:.2f} MB")
print(f"\nTotal adapter size: {total_size:.2f} MB")

# Create a zip file in /kaggle/working for easy download
zip_path = "/kaggle/working/lora_adapter"
shutil.make_archive(zip_path, "zip", adapter_dir)
print(f"\nDownloadable zip: {zip_path}.zip")
print("Download from: Kaggle Output tab > lora_adapter.zip")

In [ ]:
# ==========================================
# 10. Quick Validation Inference
# ==========================================

model.eval()

# Test on a validation sample
val_sample = val_dataset[0]
inputs = {k: v.unsqueeze(0).to(model.device) for k, v in val_sample.items()}

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        max_new_tokens=256,
        do_sample=False,
    )

generated = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)[0]

print("Validation sample prediction:")
print(generated)
print("\nGround truth:")
print(val_dataset.data[0]["conversations"][1]["value"])

In [ ]:
# ==========================================
# 11. Training Summary
# ==========================================

print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model:          {config.model_id}")
print(f"LoRA rank:      {config.lora_rank}")
print(f"LoRA alpha:     {config.lora_alpha}")
print(f"Batch size:     {config.per_device_batch_size} × {config.gradient_accumulation_steps} = {config.per_device_batch_size * config.gradient_accumulation_steps}")
print(f"Learning rate:  {config.learning_rate}")
print(f"Epochs:         {config.num_epochs}")
print(f"Frames/clip:    {config.frames_per_clip}")
print(f"Training loss:  {train_result.training_loss:.4f}")
print(f"Peak VRAM:      {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"Adapter saved:  {adapter_dir}")
print("=" * 60)